# EDA: what drives coral change on Malaysian reefs, and where can local action help?

Reef Check Malaysia surveys, 2015–2025, with NOAA Coral Reef Watch heat stress.

| Section | Question |
|---|---|
| 1 | What happened to coral cover and heat stress, 2015–2025? |
| 2 | After heat stress and crown-of-thorns, how much variation in coral cover goes with locally manageable pressures? |
| 3 | Which sites lose coral mainly outside heat years (local action worthwhile) vs mainly after heat stress? |
| 4 | Did reefs under heavy boat pressure recover more during the 2020–2022 travel shutdown? |
| 5 | Patterns behind the forecasting model: pull toward the regional average, heat → decline, survey noise |
| 6 | What does 2026 heat stress mean for the next surveys? |

**Definitions used throughout**
- **Change** = live coral cover at a survey − cover at the island's previous survey (percentage points).
- **Heat-exposed change**: peak heat stress in the year before the survey was ≥ 4 degree-heating weeks (DHW), NOAA's level where significant bleaching is expected.
- **Pollution indicators** are expressed as a share of the non-coral seabed (%), so they aren't mechanically tied to coral cover (coral + the substrate groups always add to 100%).
- **Boat-pressure index** (stand-in for tourism / development intensity, which isn't in the data): the share of an island's 2015–2019 surveys that reported anchor damage. Defined for the 28 islands with at least 3 surveys in 2015–2019; "high" = above the median. It measures boat traffic, not tourism itself.
- **Uncertainty**: 95% intervals from resampling whole islands (an island's surveys aren't independent).

These are associations in observational data, not proven causes.

In [ ]:
from itertools import combinations
from math import factorial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titleweight': 'bold', 'axes.titlesize': 11})

SEED = 42
N_BOOT = 1000
YEARS = (2015, 2025)
HEAT_DHW = 4.0
C = {'thermal': '#d1495b', 'cot': '#edae49', 'local': '#00798c', 'start': '#8d99ae', 'rest': '#e9ecef',
     'high': '#c1121f', 'low': '#669bbc', 'calm': '#00798c', 'heat': '#d1495b', 'grey': '#6c757d'}
REGION = {'Johor': 'Peninsular east', 'Pahang': 'Peninsular east', 'Terengganu': 'Peninsular east',
          'Kedah': 'Peninsular west', 'Perak': 'Peninsular west', 'Malacca': 'Peninsular west',
          'Negeri Sembilan': 'Peninsular west', 'Sabah': 'Sabah', 'Sarawak': 'Sarawak'}
GRP = ['grp_other', 'grp_available_substrate', 'grp_sand', 'grp_disturbance_indicators', 'grp_pollution_indicators']

raw = pd.read_csv('master_reef_tourism_dataset.csv').sort_values(['island', 'survey_year']).reset_index(drop=True)
raw['region'] = raw['state'].map(REGION)
noncoral = raw[GRP].sum(axis=1, min_count=len(GRP))
raw['pollution_share'] = raw['grp_pollution_indicators'] / noncoral * 100

# Previous-survey values (computed on the full history so 2015 rows see their 2014 survey)
by_island = raw.groupby('island')
for col in ['impact_anchor', 'impact_trash', 'pollution_share', 'inv_crown_of_thorns', 'cot_outbreak',
            'island_vs_region_pct']:
    raw[f'prev_{col}'] = by_island[col].shift(1)
raw['heat_exposed'] = raw['dhw_peak_prev_year'] >= HEAT_DHW

# Boat-pressure index: share of 2015-2019 surveys reporting anchor damage
pre = raw[raw['survey_year'].between(2015, 2019)].groupby('island')['impact_anchor'].agg(['size', 'mean'])
pre = pre[pre['size'] >= 3]['mean']
BOAT_MEDIAN = pre.median()
raw['boat_pressure'] = raw['island'].map(pre)
raw['boat_group'] = (pd.Series(np.where(raw['boat_pressure'] > BOAT_MEDIAN, 'High boat pressure', 'Low boat pressure'),
                               index=raw.index).where(raw['boat_pressure'].notna()))

lv = raw[raw['survey_year'].between(*YEARS)].copy()        # every survey (coral level)
ch = lv.dropna(subset=['lcc_change']).copy()                # surveys with a previous survey (coral change)
print(f'{len(lv)} surveys and {len(ch)} changes in {YEARS[0]}-{YEARS[1]}, {lv["island"].nunique()} islands')
print(f'boat-pressure index: {len(pre)} islands, median {BOAT_MEDIAN:.2f}; '
      f'{(pre > BOAT_MEDIAN).sum()} high / {(pre <= BOAT_MEDIAN).sum()} low')


def island_boot(df, stat, n_boot=N_BOOT, seed=SEED):
    """95% interval of stat(df) from resampling whole islands."""
    rng = np.random.default_rng(seed)
    groups = {i: g for i, g in df.groupby('island')}
    names = list(groups)
    vals = [stat(pd.concat([groups[i] for i in rng.choice(names, len(names))], ignore_index=True))
            for _ in range(n_boot)]
    return np.nanpercentile(vals, [2.5, 97.5])


def mean_ci(df, col, by):
    """Mean of col per group with island-bootstrap 95% intervals."""
    rows = []
    for key, g in df.groupby(by, observed=True):
        lo, hi = island_boot(g, lambda d: d[col].mean())
        rows.append({by if isinstance(by, str) else 'group': key, 'n': len(g), 'islands': g['island'].nunique(),
                     'mean': g[col].mean(), 'ci_low': lo, 'ci_high': hi})
    return pd.DataFrame(rows)

## 1. What happened to coral cover and heat stress, 2015–2025?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
cover = lv.groupby(['survey_year', 'region'])['live_coral_cover_pct'].mean().unstack()
for region, color in zip(cover.columns, ['#00798c', '#edae49', '#d1495b', '#6a4c93']):
    axes[0].plot(cover.index, cover[region], marker='o', ms=4, color=color, label=region)
axes[0].plot(cover.index, lv.groupby('survey_year')['live_coral_cover_pct'].mean(), color='black', lw=2.5,
             label='All sites')
axes[0].set_title('Mean live coral cover by region')
axes[0].set_ylabel('Live coral cover (%)')
axes[0].legend(fontsize=8, ncol=2)

heat = (raw.groupby(['survey_year', 'noaa_station_id'])['noaa_max_dhw'].first().unstack()
        .loc[YEARS[0]:YEARS[1], ['sabah', 'singapore', 'malacca_strait', 'northern_borneo']])
heat.plot.bar(ax=axes[1], width=0.8, color=['#d1495b', '#00798c', '#edae49', '#6a4c93'])
axes[1].axhline(HEAT_DHW, color='black', ls='--', lw=1)
axes[1].text(-0.4, HEAT_DHW + 0.3, 'significant bleaching expected (4 DHW)', fontsize=8)
axes[1].set_title('Peak heat stress per year by NOAA station')
axes[1].set_ylabel('Peak DHW (°C-weeks)')
axes[1].set_xlabel('')
axes[1].legend(fontsize=8, title=None)
plt.tight_layout()
plt.show()

summary = lv.groupby('survey_year').agg(surveys=('island', 'size'), mean_cover=('live_coral_cover_pct', 'mean'))
summary['mean_change'] = ch.groupby('survey_year')['lcc_change'].mean()
summary.round(2).T

**Takeaway (1).** Average coral cover slid from 47% (2015) to about 41% (2018–2020), recovered to 48% by 2022, then fell to **40% in 2025, the lowest of the period**. The two regional heat events line up with the declines: 2016 (Sabah and Malacca Strait above 10 DHW) and 2024 (all four stations above 4 DHW, up to 13.6), followed by the worst single year for coral, 2025 (−5.0 points on average). Sabah has the lowest cover every year (31–41%) and the most frequent heat stress: above 4 DHW in 7 of the 11 years.

## 2. After heat stress and crown-of-thorns, how much variation goes with locally manageable pressures?

The variation explained (R²) by a linear model is split between blocks of predictors with a **Shapley decomposition**: each block's share is its average contribution over every order in which the blocks can be added, so no block gets credit just for being added first.

| Block | Coral **level** (why some reefs have more coral) | Coral **change** (why reefs gain or lose) |
|---|---|---|
| Starting cover | – | cover at the previous survey (captures the pull toward the average) |
| Thermal stress | survey-year and previous-year peak DHW | same |
| Crown-of-thorns | density and outbreak flag at the survey | at the previous survey |
| **Local pressures** | anchor damage at the survey, pollution-indicator share, boat-pressure index | at the previous survey, plus the boat-pressure index |

Sample: the 28 islands with a boat-pressure index. Adding any 3 predictors raises R² a little by chance, so the local block is also compared with a **permutation null**: shuffle the local pressures across surveys 500 times and see how large their share gets by luck.

In [ ]:
LEVEL_BLOCKS = {'Thermal stress': ['noaa_max_dhw', 'dhw_peak_prev_year'],
                'Crown-of-thorns': ['inv_crown_of_thorns', 'cot_outbreak'],
                'Local pressures': ['impact_anchor', 'pollution_share', 'boat_pressure']}
CHANGE_BLOCKS = {'Starting cover': ['lcc_prev'],
                 'Thermal stress': ['dhw_peak_prev_year', 'noaa_max_dhw'],
                 'Crown-of-thorns': ['prev_inv_crown_of_thorns', 'prev_cot_outbreak'],
                 'Local pressures': ['prev_impact_anchor', 'prev_pollution_share', 'boat_pressure']}


def r2(X, y):
    if X.shape[1] == 0:
        return 0.0
    X1 = np.column_stack([np.ones(len(y)), X])
    res = y - X1 @ np.linalg.lstsq(X1, y, rcond=None)[0]
    return 1 - res @ res / ((y - y.mean()) @ (y - y.mean()))


def shapley(blocks, y):
    """Shapley share of R² per block; blocks maps name -> 2D array."""
    names, B, cache = list(blocks), len(blocks), {}

    def R(S):
        key = frozenset(S)
        if key not in cache:
            cache[key] = r2(np.column_stack([blocks[n] for n in S]) if S else np.empty((len(y), 0)), y)
        return cache[key]

    out = {}
    for j in names:
        others = [n for n in names if n != j]
        out[j] = sum(factorial(k) * factorial(B - k - 1) / factorial(B) * (R(S + (j,)) - R(S))
                     for k in range(B) for S in combinations(others, k))
    return out, R(tuple(names))


def decompose(df, blocks, target):
    cols = [c for cs in blocks.values() for c in cs]
    d = df.dropna(subset=cols + [target]).reset_index(drop=True)
    y = d[target].to_numpy(float)
    arrays = {n: d[cs].to_numpy(float) for n, cs in blocks.items()}
    shares, total = shapley(arrays, y)

    rng = np.random.default_rng(SEED)
    rows_of = {i: np.flatnonzero(d['island'].to_numpy() == i) for i in d['island'].unique()}
    boot = []
    for _ in range(N_BOOT):
        idx = np.concatenate([rows_of[i] for i in rng.choice(list(rows_of), len(rows_of))])
        boot.append(shapley({n: a[idx] for n, a in arrays.items()}, y[idx])[0])
    boot = pd.DataFrame(boot)

    null = []
    for _ in range(500):
        perm = rng.permutation(len(y))
        null.append(shapley({**arrays, 'Local pressures': arrays['Local pressures'][perm]}, y)[0]['Local pressures'])
    table = pd.DataFrame({'share_of_variation': shares, 'ci_low': boot.quantile(0.025), 'ci_high': boot.quantile(0.975)})
    return table, total, np.percentile(null, 95), np.mean(np.array(null) >= shares['Local pressures']), d


level_tab, level_r2, level_null95, level_p, level_d = decompose(lv, LEVEL_BLOCKS, 'live_coral_cover_pct')
change_tab, change_r2, change_null95, change_p, change_d = decompose(ch, CHANGE_BLOCKS, 'lcc_change')
for name, tab, tot, n95, p, d in [('LEVEL', level_tab, level_r2, level_null95, level_p, level_d),
                                   ('CHANGE', change_tab, change_r2, change_null95, change_p, change_d)]:
    print(f'--- Coral {name}: n={len(d)} surveys, {d["island"].nunique()} islands; total R² {tot:.3f}; '
          f'local block: chance level (95th pct of null) {n95:.3f}, permutation p = {p:.3f}')
    display(tab.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={'width_ratios': [3, 2]})
order = ['Starting cover', 'Thermal stress', 'Crown-of-thorns', 'Local pressures']
colors = {'Starting cover': C['start'], 'Thermal stress': C['thermal'], 'Crown-of-thorns': C['cot'],
          'Local pressures': C['local']}
for row, (label, tab, tot) in enumerate([('Coral change', change_tab, change_r2), ('Coral level', level_tab, level_r2)]):
    left = 0
    for block in order:
        if block not in tab.index:
            continue
        v = tab.loc[block, 'share_of_variation'] * 100
        axes[0].barh(row, v, left=left, color=colors[block], label=block if row == 0 or block not in change_tab.index else None)
        if v >= 4:
            axes[0].text(left + v / 2, row, f'{v:.1f}', ha='center', va='center', color='white', fontsize=9, fontweight='bold')
        left += v
    axes[0].barh(row, 100 - left, left=left, color=C['rest'], label='Unexplained' if row == 0 else None)
    axes[0].text(left + 1, row, f'unexplained {100 - left:.0f}%', va='center', fontsize=9, color=C['grey'])
axes[0].set_yticks([0, 1], ['Coral change', 'Coral level'])
axes[0].set_xlim(0, 100)
axes[0].set_xlabel('% of variation (Shapley share of R²)')
axes[0].set_title('How much variation each block of predictors accounts for')
axes[0].legend(fontsize=8, ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.25))

for row, (tab, n95) in enumerate([(change_tab, change_null95), (level_tab, level_null95)]):
    v, lo, hi = tab.loc['Local pressures', ['share_of_variation', 'ci_low', 'ci_high']] * 100
    axes[1].errorbar(v, row, xerr=[[v - lo], [hi - v]], fmt='o', color=C['local'], capsize=4, ms=8)
    axes[1].plot([n95 * 100] * 2, [row - 0.25, row + 0.25], color='black', lw=2)
axes[1].plot([], [], color='black', lw=2, label='chance level (95th pct of shuffled)')
axes[1].set_yticks([0, 1], ['Coral change', 'Coral level'])
axes[1].set_ylim(-0.6, 1.6)
axes[1].set_xlabel('% of variation, 95% interval')
axes[1].set_title('Local pressures vs chance')
axes[1].legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.show()

### Direction and size of each local pressure

Coefficients from the full model (all blocks), in natural units, with island-bootstrap 95% intervals. Anchor damage: reported vs not. Pollution indicators: per +10 points of the non-coral seabed. Boat pressure: per +0.5, e.g. an island going from anchor damage in 20% of surveys to 70%.

In [ ]:
def coefs(d, blocks, target):
    cols = [c for cs in blocks.values() for c in cs]
    X, y = d[cols].to_numpy(float), d[target].to_numpy(float)
    fit = lambda X, y: np.linalg.lstsq(np.column_stack([np.ones(len(y)), X]), y, rcond=None)[0][1:]
    b = fit(X, y)
    rng = np.random.default_rng(SEED)
    rows_of = {i: np.flatnonzero(d['island'].to_numpy() == i) for i in d['island'].unique()}
    boot = np.array([fit(X[idx], y[idx]) for idx in
                     (np.concatenate([rows_of[i] for i in rng.choice(list(rows_of), len(rows_of))]) for _ in range(N_BOOT))])
    return pd.DataFrame({'coef': b, 'ci_low': np.percentile(boot, 2.5, axis=0), 'ci_high': np.percentile(boot, 97.5, axis=0)}, index=cols)


SCALE = {'impact_anchor': 1, 'prev_impact_anchor': 1, 'pollution_share': 10, 'prev_pollution_share': 10, 'boat_pressure': 0.5}
LABEL = {'impact_anchor': 'Anchor damage reported', 'prev_impact_anchor': 'Anchor damage reported (prev. survey)',
         'pollution_share': 'Pollution indicators +10 pts', 'prev_pollution_share': 'Pollution indicators +10 pts (prev. survey)',
         'boat_pressure': 'Boat-pressure index +0.5'}
fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))
local_effects = {}
for ax, (label, d, blocks, target) in zip(axes, [('Coral level (% cover)', level_d, LEVEL_BLOCKS, 'live_coral_cover_pct'),
                                                ('Coral change (points)', change_d, CHANGE_BLOCKS, 'lcc_change')]):
    t = coefs(d, blocks, target).loc[blocks['Local pressures']]
    t = t.mul([SCALE[c] for c in t.index], axis=0)
    local_effects[label] = t
    ypos = np.arange(len(t))[::-1]
    for y0, (col, r) in zip(ypos, t.iterrows()):
        color = C['local'] if (r.ci_low > 0 or r.ci_high < 0) else C['grey']
        ax.errorbar(r.coef, y0, xerr=[[r.coef - r.ci_low], [r.ci_high - r.coef]], fmt='o', color=color, capsize=3)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_yticks(ypos, [LABEL[c] for c in t.index])
    ax.set_title(label)
axes[0].set_xlabel('difference in live coral cover (points)')
axes[1].set_xlabel('difference in next change (points)')
fig.suptitle('Local pressures after accounting for heat and crown-of-thorns (teal = interval excludes 0)', fontsize=10)
plt.tight_layout()
plt.show()
pd.concat(local_effects).round(2)

**Takeaway (2).** Local pressures matter for **how much coral a reef has**, but barely register in **year-to-year change**.
- **Coral level:** after heat stress and crown-of-thorns, local pressures account for about **7% of the variation** (95% interval 1–24%), well above chance (3%, permutation p = 0.002). The signal is chronic boat pressure: islands where anchor damage is reported habitually carry about **6 points less coral** per +0.5 on the boat-pressure index (interval −12.5 to −0.2). Anchor damage at a single survey and pollution indicators show no clear link. Thermal stress's larger share (16%) is partly regional: Sabah is both hotter and lower in cover.
- **Coral change:** local pressures account for about 3%, which is **indistinguishable from chance** (p = 0.07). Starting cover (the pull toward the average) accounts for the most, 10%. High boat pressure leans toward bigger declines (−1.7 points per +0.5, interval −3.8 to 0.0). Pollution indicators come out *positive* here (+1.2 per +10 points), but that isn't robust: Part B of the modelling notebook, with fuller controls, finds 0.0 (−1.7 to +1.5).
- **Most variation is unexplained** (74% of level, 84% of change), and survey noise is a large part of it (section 5d).

## 3. Which sites lose coral mainly outside heat years, and which mainly after heat stress?

For every island with at least 5 changes in 2015–2025 (including at least 1 heat-exposed and 2 calm ones), each change is split by whether the previous year had heat stress ≥ 4 DHW:
- **Calm-year change**: summed change over intervals without significant heat.
- **Heat-year change**: summed change over heat-exposed intervals.

Sites that lose coral mainly in **calm years** are the candidates where local pressures (anchoring, waste, pollution, crown-of-thorns) plausibly matter and **local intervention is worthwhile**. Sites that lose mainly **after heat** are driven by regional thermal stress; local action can support recovery but can't prevent the losses. The local-pressure columns show the evidence behind each site.

Caveat: sums over a few noisy surveys. Sites surveyed with ≤ 3 sites (flagged) swing by 20–30 points from one survey to the next.

In [ ]:
g = ch.groupby('island')
sites = pd.DataFrame({
    'region': g['region'].first(),
    'changes': g.size(),
    'heat_changes': g['heat_exposed'].sum(),
    'calm_change_sum': ch[~ch['heat_exposed']].groupby('island')['lcc_change'].sum(),
    'heat_change_sum': ch[ch['heat_exposed']].groupby('island')['lcc_change'].sum(),
    'calm_change_per_survey': ch[~ch['heat_exposed']].groupby('island')['lcc_change'].mean(),
    'heat_change_per_survey': ch[ch['heat_exposed']].groupby('island')['lcc_change'].mean(),
    'anchor_share': lv.groupby('island')['impact_anchor'].mean(),
    'trash_share': lv.groupby('island')['impact_trash'].mean(),
    'pollution_share': lv.groupby('island')['pollution_share'].mean(),
    'cot_outbreak_share': lv.groupby('island')['cot_outbreak'].mean(),
    'median_sites': lv.groupby('island')['n_sites'].median(),
    'boat_group': g['boat_group'].first(),
}).fillna({'calm_change_sum': 0, 'heat_change_sum': 0})
sites = sites[(sites['changes'] >= 5) & (sites['heat_changes'] >= 1) & (sites['changes'] - sites['heat_changes'] >= 2)]
sites['net_change'] = sites['calm_change_sum'] + sites['heat_change_sum']
calm_loss, heat_loss = -sites['calm_change_sum'].clip(upper=0), -sites['heat_change_sum'].clip(upper=0)
sites['pattern'] = np.select(
    [sites['net_change'] >= 0, calm_loss > heat_loss],
    ['Stable or gaining', 'Losses mainly in calm years (local action worthwhile)'],
    'Losses mainly after heat (thermal)')
sites['low_reliability'] = sites['median_sites'] <= 3
sites = sites.sort_values('net_change')
print(sites['pattern'].value_counts().to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 0.32 * len(sites) + 1.5))
ypos = np.arange(len(sites))
ax.barh(ypos, sites['calm_change_sum'], color=C['calm'], label='change in calm years')
ax.barh(ypos, sites['heat_change_sum'], left=np.where((sites['calm_change_sum'] < 0) == (sites['heat_change_sum'] < 0),
                                                       sites['calm_change_sum'], 0),
        color=C['heat'], alpha=0.85, label='change after heat stress (≥ 4 DHW)')
ax.scatter(sites['net_change'], ypos, color='black', zorder=3, s=18, label='net change 2015–2025')
labels = [f'{i}{" *" if r.low_reliability else ""}' for i, r in sites.iterrows()]
ax.set_yticks(ypos, labels, fontsize=8)
ax.axvline(0, color='black', lw=0.8)
for y0, p in zip(ypos, sites['pattern']):
    tag = {'Stable or gaining': '', 'Losses mainly in calm years (local action worthwhile)': 'LOCAL',
           'Losses mainly after heat (thermal)': 'HEAT'}[p]
    ax.text(ax.get_xlim()[1], y0, tag, va='center', ha='right', fontsize=7,
            color=C['calm'] if tag == 'LOCAL' else C['heat'], fontweight='bold')
ax.set_xlabel('summed change in live coral cover, 2015–2025 (points)')
ax.set_title('Where did each site lose coral: in calm years or after heat stress?  (* = surveys usually ≤ 3 sites)')
ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.06), ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for grp, color in [('High boat pressure', C['high']), ('Low boat pressure', C['low']), (None, '#adb5bd')]:
    s = sites[sites['boat_group'].isna()] if grp is None else sites[sites['boat_group'] == grp]
    ax.scatter(s['heat_change_per_survey'], s['calm_change_per_survey'], s=s['changes'] * 12, color=color, alpha=0.75,
               edgecolor='white', label=grp or 'no boat-pressure index')
for i, r in sites.iterrows():
    ax.annotate(i, (r['heat_change_per_survey'], r['calm_change_per_survey']), fontsize=7, xytext=(3, 3),
                textcoords='offset points')
ax.axhline(0, color='black', lw=0.8)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('average change per survey after heat stress (points)')
ax.set_ylabel('average change per survey in calm years (points)')
ax.set_title('Site typology: bottom half = losing coral even without heat stress')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

cols = ['region', 'pattern', 'net_change', 'calm_change_sum', 'heat_change_sum', 'anchor_share', 'trash_share',
        'pollution_share', 'cot_outbreak_share', 'median_sites', 'boat_group']
sites.loc[sites['pattern'] != 'Stable or gaining', cols].round(2)

**Takeaway (3).** Of 28 sites with enough history: **10 lost coral mainly after heat stress, 9 mainly in calm years, and 9 were stable or gaining.**
- **Local-action candidates** (losses in calm years): **Redang** (−23.5 points in calm years; trash reported at 91% of surveys and crown-of-thorns outbreaks at 64%) and **Tioman** (anchor damage 91%, trash 91%, crown-of-thorns 55%) carry the clearest local evidence. **Pemanggil, Tinggi and Aur & Dayang** pair calm-year losses with high pollution-indicator cover (21–25% of the non-coral seabed) and frequent crown-of-thorns outbreaks (55–75%). Crown-of-thorns removal is itself a local intervention, so these sites are where management can plausibly change the outcome. Most are in the Peninsular east (Johor, Pahang, Terengganu).
- **Thermally driven** (losses after heat): mostly Sabah. **Labuan** (−32.9 after heat, +0.9 in calm years), **Usukan Cove, Mataking, Tunku Abdul Rahman Park and Mantanani**, plus **Perhentian** on the east coast. Local measures here support recovery but can't prevent the losses.
- **Treat this as a screen, not a diagnosis.** Sipadan's calm-year losses come with no anchor damage at all. Penyu's pattern is two big offsetting swings (−24 in calm years, +20 after heat). Pangkor Laut is surveyed at a single site, so its numbers are mostly noise.

## 4. Did reefs under heavy boat pressure recover more during the 2020–2022 shutdown?

Malaysia's movement-control order began in March 2020, and international borders reopened in April 2022. Changes measured at the 2020, 2021 and 2022 surveys cover this period. The comparison is between the 28 islands with a boat-pressure index, split at the median:
1. **Did the proxy actually drop?** The share of surveys reporting anchor damage by year.
2. **Coral change before (2017–19), during (2020–22) and after (2023–25)** the shutdown, by group.
3. **Difference-in-differences:** (shutdown − before) for high-pressure islands minus the same for low-pressure islands. A positive value means high-pressure reefs improved more during the shutdown. It's shown raw and *adjusted*, where each change is first corrected for starting cover and heat stress (so a hot year in one group doesn't masquerade as a shutdown effect).

In [ ]:
sub = lv[lv['boat_group'].isin(['High boat pressure', 'Low boat pressure'])]
anchor_rate = sub.groupby(['survey_year', 'boat_group'])['impact_anchor'].mean().unstack()

PERIODS = {'Before\n2017–19': (2017, 2019), 'Shutdown\n2020–22': (2020, 2022), 'After\n2023–25': (2023, 2025)}
cs = ch[ch['boat_group'].isin(['High boat pressure', 'Low boat pressure'])].copy()
labels = np.select([cs['survey_year'].between(*v) for v in PERIODS.values()], list(PERIODS), default='')
cs['period'] = pd.Categorical(np.where(labels == '', None, labels), categories=list(PERIODS))
cs = cs.dropna(subset=['period'])

# adjusted change: residual after starting cover and heat stress (fitted on all 2015-2025 changes of these islands)
X = np.column_stack([np.ones(len(cs)), cs[['lcc_prev', 'dhw_peak_prev_year', 'noaa_max_dhw']].to_numpy(float)])
cs['adj_change'] = cs['lcc_change'] - X @ np.linalg.lstsq(X, cs['lcc_change'].to_numpy(float), rcond=None)[0] + cs['lcc_change'].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for grp, color in [('High boat pressure', C['high']), ('Low boat pressure', C['low'])]:
    axes[0].plot(anchor_rate.index, anchor_rate[grp] * 100, marker='o', color=color, label=grp)
axes[0].axvspan(2019.5, 2022.5, color='#adb5bd', alpha=0.25, label='shutdown surveys')
axes[0].set_ylabel('% of surveys reporting anchor damage')
axes[0].set_title('1. Did anchor damage drop during the shutdown?')
axes[0].legend(fontsize=8)

period_stats = []
for k, (grp, color) in enumerate([('High boat pressure', C['high']), ('Low boat pressure', C['low'])]):
    t = mean_ci(cs[cs['boat_group'] == grp], 'lcc_change', 'period')
    t['boat_group'] = grp
    period_stats.append(t)
    x = np.arange(len(t)) + (k - 0.5) * 0.25
    axes[1].errorbar(x, t['mean'], yerr=[t['mean'] - t['ci_low'], t['ci_high'] - t['mean']], fmt='o', color=color,
                     capsize=4, ms=8, label=grp)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_xticks(range(len(PERIODS)), list(PERIODS))
axes[1].set_ylabel('mean change per survey (points), 95% CI')
axes[1].set_title('2. Coral change before, during and after the shutdown')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


def did(d, col):
    m = d.groupby(['boat_group', 'period'], observed=True)[col].mean()
    get = lambda grp, k: m.get((grp, PERIODS_KEYS[k]), np.nan)   # a resample can miss a group-period cell
    return ((get('High boat pressure', 1) - get('High boat pressure', 0))
            - (get('Low boat pressure', 1) - get('Low boat pressure', 0)))


PERIODS_KEYS = list(PERIODS)
did_table = pd.DataFrame({col: {'difference_in_differences': did(cs, col), **dict(zip(['ci_low', 'ci_high'],
                               island_boot(cs, lambda d, c=col: did(d, c))))}
                          for col in ['lcc_change', 'adj_change']}).T.rename(index={'lcc_change': 'raw', 'adj_change': 'adjusted for cover & heat'})
display(pd.concat(period_stats).set_index(['boat_group', 'period']).round(2))
heat_by_period = cs.groupby(['boat_group', 'period'], observed=True)['dhw_peak_prev_year'].mean().unstack().round(2)
print('mean previous-year heat stress (DHW) per group and period:')
display(heat_by_period)
did_table.round(2)

**Takeaway (4).** **Suggestive, not demonstrated.**
- **The proxy responded to the shutdown.** At high-pressure islands, anchor damage was reported at 64% of surveys in 2019, **33% in 2020**, and back to 64% by 2022. At low-pressure islands it was already 0% in 2019 and 2020, so there was nothing to fall; it returned to 20–42% from 2021.
- **High-pressure reefs did improve more.** Their average change went from −1.0 before to **+1.4 during** the shutdown (+2.4), against −0.5 to +0.6 (+1.0) at low-pressure reefs. The difference-in-differences is +1.4 points per survey (+1.0 after adjusting for starting cover and heat), but the 95% interval spans roughly −4.5 to +7, so the data can't separate it from chance.
- **After reopening, high-pressure reefs declined more** (−3.4 vs −2.2 per survey), consistent with pressure returning, though this coincides with the 2024 heat event.
- **Heat was lower for both groups during the shutdown**, which is why the adjusted comparison matters.

## 5. The patterns behind the forecasting model

The forecasting model (weighted Lasso) leans on three things. These charts show each one directly in the data.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))

# 5a pull toward the regional average
d = ch.dropna(subset=['prev_island_vs_region_pct']).copy()
d['bin'] = pd.cut(d['prev_island_vs_region_pct'], [-np.inf, -15, -5, 5, 15, np.inf],
                  labels=['< −15', '−15 to −5', '−5 to +5', '+5 to +15', '> +15'])
t = mean_ci(d, 'lcc_change', 'bin')
axes[0, 0].bar([f'{b}\n(n={n})' for b, n in zip(t['bin'], t['n'])], t['mean'],
               yerr=[t['mean'] - t['ci_low'], t['ci_high'] - t['mean']],
               color=[C['calm'] if v > 0 else C['heat'] for v in t['mean']], capsize=4)
axes[0, 0].axhline(0, color='black', lw=0.8)
axes[0, 0].set_xlabel('how far above its regional average the reef was at the last survey (points)')
axes[0, 0].set_ylabel('mean next change (points)')
axes[0, 0].set_title('a. Reefs drift back toward their regional average')
reversion = t

# 5b heat dose-response
d = ch.copy()
d['band'] = pd.cut(d['dhw_peak_prev_year'], [-0.01, 1, 4, 8, np.inf], labels=['< 1', '1–4', '4–8', '≥ 8'])
t = mean_ci(d, 'lcc_change', 'band')
axes[0, 1].bar([f'{b}\n(n={n})' for b, n in zip(t['band'], t['n'])], t['mean'],
               yerr=[t['mean'] - t['ci_low'], t['ci_high'] - t['mean']],
               color=['#fcd5ce', '#f8ad9d', '#f07167', '#b5172a'], capsize=4)
axes[0, 1].axhline(0, color='black', lw=0.8)
axes[0, 1].set_xlabel('peak heat stress in the year before the survey (DHW)')
axes[0, 1].set_ylabel('mean change (points)')
axes[0, 1].set_title('b. More heat last year → more coral lost')
dose = t

# 5c year comparison
yr = mean_ci(ch, 'lcc_change', 'survey_year')
heat_prev = ch.groupby('survey_year')['dhw_peak_prev_year'].mean()
cmap = plt.get_cmap('Reds')
axes[1, 0].bar(yr['survey_year'], yr['mean'], yerr=[yr['mean'] - yr['ci_low'], yr['ci_high'] - yr['mean']],
               color=[cmap(0.2 + 0.7 * min(h / 8, 1)) for h in heat_prev.loc[yr['survey_year']]], capsize=3)
axes[1, 0].axhline(0, color='black', lw=0.8)
axes[1, 0].set_xticks(yr['survey_year'])
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].set_ylabel('mean change (points), 95% CI')
axes[1, 0].set_title('c. Year by year: darker = more heat the year before')

# 5d survey noise
d = lv.dropna(subset=['lcc_change', 'n_sites']).copy()
d['abs_change'] = d['lcc_change'].abs()
d['sites'] = pd.cut(d['n_sites'], [0, 3, 5, 8, 40], labels=['1–3', '4–5', '6–8', '9+'])
t = mean_ci(d, 'abs_change', 'sites')
axes[1, 1].bar([f'{s}\n(n={n})' for s, n in zip(t['sites'], t['n'])], t['mean'],
               yerr=[t['mean'] - t['ci_low'], t['ci_high'] - t['mean']], color=C['grey'], capsize=4)
axes[1, 1].set_xlabel('sites surveyed')
axes[1, 1].set_ylabel('typical size of a change, |change| (points)')
axes[1, 1].set_title('d. Small surveys swing much more (survey noise)')
plt.tight_layout()
plt.show()
display(reversion.round(2), dose.round(2))

**Takeaway (5).**
- **a. Pull toward the regional average:** reefs more than 15 points above their region lost **6.3 points** on average at the next survey, while those more than 15 below gained **2.4**. This is the forecasting model's strongest feature.
- **b. Heat stress:** after a year below 4 DHW, coral held steady (−0.3 and −0.1). After 4–8 DHW it fell 1.5 points, and after 8 or more DHW, **3.2 points**. That's why previous-year heat is the one external driver the model keeps.
- **c. Year by year:** the two years after the hottest ones had big declines: 2017 (−1.5, after 9.1 DHW on average) and 2025 (−5.0, after 10.5). Most gains follow cool years (2019 and 2022, after about 1 DHW). The exception is 2021, which gained 1.7 despite moderate heat (4.5).
- **d. Survey noise:** surveys covering 1–3 sites swing by **13.9 points** on average, against 5.7–6.4 for larger surveys. This is why the model down-weights small surveys, and part of why so much variation stays unexplained.

## 6. What does 2026 heat stress mean for the next surveys?

Previous-year heat is the one external driver the forecasting model keeps, so 2026 heat stress points to what the 2027 surveys may show. NOAA regional station data run to 10 September 2026, so the 2026 bars can still rise.

In [ ]:
COLS = ['y', 'm', 'd', 'sst_min', 'sst_max', 'sst90', 'ssta90', 'hs90', 'dhw', 'baa']
peaks = {}
for station in ['sabah', 'singapore', 'malacca_strait', 'northern_borneo']:
    lines = Path(f'data/raw/structured/noaa_crw/{station}.txt').read_text().splitlines()
    start = next(i for i, l in enumerate(lines) if l.startswith('YYYY'))
    s = pd.read_csv(Path(f'data/raw/structured/noaa_crw/{station}.txt'), sep=r'\s+', skiprows=start + 1, names=COLS)
    peaks[station] = s.groupby('y')['dhw'].max()
peaks = pd.DataFrame(peaks)
islands_per_station = raw.groupby('noaa_station_id')['island'].nunique()

fig, ax = plt.subplots(figsize=(11, 4))
show_years = [2016, 2019, 2020, 2023, 2024, 2025, 2026]
peaks.loc[show_years].T.plot.bar(ax=ax, width=0.85, color=['#f8ad9d', '#f4978e', '#f08080', '#e5383b', '#ba181b', '#dee2e6', '#161a1d'])
ax.axhline(HEAT_DHW, color='black', ls='--', lw=1)
ax.set_xticklabels([f'{s}\n({islands_per_station.get(s, 0)} islands)' for s in peaks.columns], rotation=0)
ax.set_ylabel('peak DHW (°C-weeks)')
ax.set_title('Peak heat stress by station: 2026 (black, to 10 Sep) against past bleaching years')
ax.legend(title=None, fontsize=8, ncol=7, loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.tight_layout()
plt.show()

# what the heat dose-response implies for 2027 surveys, per station
band_mean = dose.set_index('band')['mean']
outlook = pd.DataFrame({'peak_dhw_2026_to_date': peaks.loc[2026],
                        'islands': islands_per_station.reindex(peaks.columns).fillna(0).astype(int)})
outlook['dhw_band'] = pd.cut(outlook['peak_dhw_2026_to_date'], [-0.01, 1, 4, 8, np.inf], labels=['< 1', '1–4', '4–8', '≥ 8'])
outlook['historical_mean_change_in_band'] = outlook['dhw_band'].map(band_mean).astype(float)
outlook.round(2)

**Takeaway (6).** By 10 September 2026, **Sabah** has reached **13.4 DHW**, above its 2024 peak (11.4) and the highest year shown. The **Malacca Strait** is at 9.0, Northern Borneo 5.3 and the Singapore Strait 4.4, and the season isn't over. Historically, a year of 8 DHW or more was followed by an average **3.2-point decline** at the next survey. So the 2027 surveys should expect clear losses across Sabah's 24 islands, already the most thermally driven sites in section 3, and on the west coast.

## Summary

1. **Heat is the dominant regional driver.** The biggest declines follow the hottest years (2017 after 2016, 2025 after 2024), and each step up in previous-year heat stress means a bigger loss.
2. **Local pressure shows up in how much coral a reef has, not in year-to-year change.** Islands with habitual anchor damage carry about 6 points less coral, but local pressures add nothing beyond chance to yearly change once heat and starting cover are accounted for.
3. **Where local action is worth prioritising:** Redang, Tioman and the Johor islands (Pemanggil, Tinggi, Aur & Dayang) lose coral in years *without* heat stress, alongside anchor damage, waste, pollution indicators and crown-of-thorns outbreaks. Sabah sites (Labuan, Usukan Cove, Mataking, Tunku Abdul Rahman Park) lose it mainly after heat.
4. **The shutdown hints at a tourism effect but can't prove it.** Anchor damage halved at busy islands in 2020, and their reefs improved more, but the difference is well within the noise.
5. **2026 is shaping up as Sabah's worst heat year on record**, pointing to declines at the 2027 surveys.

*All results are associations in observational data; the boat-pressure index is a stand-in for tourism and development intensity, which aren't measured in this dataset.*